In [ ]:
import glob
import json
import os
from pathlib import Path

import pandas as pd
import seaborn as sns
sns.set()

from hipe_ocrepair_scorer.cli import (
    load_jsonl,
    load_schema,
    merge_reference_hypothesis,
    find_reference_files,
    find_hypothesis_file,
    validate_jsonl_record
)
from hipe_ocrepair_scorer.ocrepair_eval import Evaluation

### Scoring functions

In [ ]:
# Only requires document_metadata and ocr_hypothesis, doesn't have to include ground_truth or ocr_postcorrection_output
schema = load_schema("../HIPE-OCRepair-scorer/data/schema/hipe-ocrepair.schema.json")  

In [ ]:
def score_folder_pair(ref_dir, hyp_dir):
    """Evaluate all matching files across reference and hypothesis directories."""
    print(f"  Reference dir:  {ref_dir}")
    print(f"  Hypothesis dir: {hyp_dir}")
    print()

    ref_files = find_reference_files(ref_dir)
    all_merged = []

    for ref_path in ref_files:
        hyp_path = find_hypothesis_file(hyp_dir, ref_path)
        if hyp_path is None:
            print(f"  [SKIP] No hypothesis file for {ref_path.name}")
            continue

        print(f"  {ref_path.name} <-> {hyp_path.name}")
        ref_records = load_jsonl(ref_path)
        hyp_records = load_jsonl(hyp_path)
        merged = merge_reference_hypothesis(ref_records, hyp_records)
        all_merged.extend(merged)

    print(f"\n  Total: {len(all_merged)} documents")

    evaluation = Evaluation(all_merged)
    results = evaluation.score_over_datasets(normalize=True)

    return results

In [ ]:
def score_file_pair(ref, hyp):
    """Evaluate a single reference/hypothesis file pair."""
    print(f"  Reference:  {ref}")
    print(f"  Hypothesis: {hyp}")
    print()

    ref_records = load_jsonl(ref)
    hyp_records = load_jsonl(hyp)
    merged = merge_reference_hypothesis(ref_records, hyp_records)

    print(f"  Merged {len(merged)} documents")

    evaluation = Evaluation(merged)
    results = evaluation.score_over_datasets(normalize=True)

    return results

In [ ]:
# if "fold_scores" not in results or "averaged_scores" not in results:
#     results_df = pd.DataFrame(results).T
# else:
#     results_df = pd.concat([pd.DataFrame(results["fold_scores"]).T, pd.DataFrame({"average": results["averaged_scores"]}).T])

In [ ]:
def tabulate_results(results_df):
    df = results_df.copy()

    df["cmer_micro_5ci"] = df["cmer_micro"].apply(lambda x: x[1])
    df["cmer_micro_95ci"] = df["cmer_micro"].apply(lambda x: x[2])
    df["cmer_micro"] = df["cmer_micro"].apply(lambda x: x[0])
        
    df["wmer_micro_5ci"] = df["wmer_micro"].apply(lambda x: x[1])
    df["wmer_micro_95ci"] = df["wmer_micro"].apply(lambda x: x[2])
    df["wmer_micro"] = df["wmer_micro"].apply(lambda x: x[0])
        
    df["cmer_macro_5ci"] = df["cmer_macro"].apply(lambda x: x[1])
    df["cmer_macro_95ci"] = df["cmer_macro"].apply(lambda x: x[2])
    df["cmer_macro"] = df["cmer_macro"].apply(lambda x: x[0])
        
    df["wmer_macro_5ci"] = df["wmer_macro"].apply(lambda x: x[1])
    df["wmer_macro_95ci"] = df["wmer_macro"].apply(lambda x: x[2])
    df["wmer_macro"] = df["wmer_macro"].apply(lambda x: x[0])
    
    df["pref_score_cmer_macro_5ci"] = df["pref_score_cmer_macro"].apply(lambda x: x[1])
    df["pref_score_cmer_macro_95ci"] = df["pref_score_cmer_macro"].apply(lambda x: x[2])
    df["pref_score_cmer_macro"] = df["pref_score_cmer_macro"].apply(lambda x: x[0])
    
    df["pref_score_wmer_macro_5ci"] = df["pref_score_wmer_macro"].apply(lambda x: x[1])
    df["pref_score_wmer_macro_95ci"] = df["pref_score_wmer_macro"].apply(lambda x: x[2])
    df["pref_score_wmer_macro"] = df["pref_score_wmer_macro"].apply(lambda x: x[0])
    
    df["pcis_cmer_macro_5ci"] = df["pcis_cmer_macro"].apply(lambda x: x[1])
    df["pcis_cmer_macro_95ci"] = df["pcis_cmer_macro"].apply(lambda x: x[2])
    df["pcis_cmer_macro"] = df["pcis_cmer_macro"].apply(lambda x: x[0])
    
    df["pcis_wmer_macro_5ci"] = df["pcis_wmer_macro"].apply(lambda x: x[1])
    df["pcis_wmer_macro_95ci"] = df["pcis_wmer_macro"].apply(lambda x: x[2])
    df["pcis_wmer_macro"] = df["pcis_wmer_macro"].apply(lambda x: x[0])
    
    return df[[
        "cmer_micro", "cmer_micro_5ci", "cmer_micro_95ci",
        "wmer_micro", "wmer_micro_5ci", "wmer_micro_95ci", 
        "cmer_macro", "cmer_macro_5ci", "cmer_macro_95ci", 
        "wmer_macro", "wmer_macro_5ci", "wmer_macro_95ci", 
        "pref_score_cmer_macro", "pref_score_cmer_macro_5ci", "pref_score_cmer_macro_95ci", 
        "pref_score_wmer_macro", "pref_score_wmer_macro_5ci", "pref_score_wmer_macro_95ci", 
        "pcis_cmer_macro", "pcis_cmer_macro_5ci", "pcis_cmer_macro_95ci", 
        "pcis_wmer_macro", "pcis_wmer_macro_5ci", "pcis_wmer_macro_95ci"
    ]]

In [ ]:
def plot_errorbars(plot_df, results_df, axes):
    """
    Plot errorbars on an existing set of axes
    Seaborn has no capacity to plot pre-calculated error bars
    HIPE-OCRepair scorer calculates it's own errors so have to manually plot
    """
    for ax in axes:
        title = ax.title.get_text()
        dataset, metric = title.split("|")
        dataset = dataset.split("=")[1].strip()
        metric = metric.split("=")[1].strip()
        subset_df = results_df.loc[dataset]
        x = subset_df[metric + "_5ci"].index
        y5, y95 = subset_df[metric + "_5ci"].values, subset_df[metric + "_95ci"].values
        vlines = subset_df[metric + "_95ci"].values - subset_df[metric + "_5ci"].values
        ax.scatter(x=x, y=y5, marker='_', c='C0')
        ax.scatter(x=x, y=y95, marker='_', c='C0')
        ax.vlines(x=x, ymin=y5, ymax=y95)
        
    # for metric, ax in zip(plot_df["metric"].unique(), axes):
    #     x = results_df[metric + "_5ci"].index
    #     y5, y95 = results_df[metric + "_5ci"].values, results_df[metric + "_95ci"].values
    #     vlines = results_df[metric + "_95ci"].values - results_df[metric + "_5ci"].values
    #     ax.scatter(x=x, y=y5, marker='_', c='C0')
    #     ax.scatter(x=x, y=y95, marker='_', c='C0')
    #     ax.vlines(x=x, ymin=y5, ymax=y95)

In [ ]:
cw_g.axes.flatten()[0].title.get_text().split("|")

### Scoring

In [ ]:
REF_DIR = Path("../data/processed/train_ref/")
HYP_DIR = Path("../data/processed/train_hyp_as_postcorr/")

In [ ]:
ocr_postcorrection_output = {
    "transcription_unit": "None",
    "ocr_postcorrection_system": "None",
    "num_tokens": -1,
    "num_chars": -1,
    "quality_report": {
        "ocr_quality_score": -1,
        "cer": -1, "wer": -1,
        "nb_char_substitutions": -1,
        "nb_char_deletions": -1,
        "nb_char_insertions": -1}
}

In [ ]:
for p in [p for p in HYP_DIR.glob("*.jsonl")]:
    print(p)
    with open(p, "r") as f:
        json_lines = f.readlines()
        records = [json.loads(l) for l in json_lines]
        
    for i, record in enumerate(records):
        validate_jsonl_record(record=record, schema=schema, filepath=p, line_num=i)
        
        if "ocr_postcorrection_output" not in record:
            print(record.keys())    
            record["ocr_postcorrection_output"] = ocr_postcorrection_output
        
        record["ocr_postcorrection_output"]["transcription_unit"] = record["ocr_hypothesis"]["transcription_unit"]

    with open(p, "w") as f:
        [f.write(json.dumps(record) + "\n") for record in records]

In [ ]:
results = score_folder_pair(REF_DIR, HYP_DIR)

In [ ]:
results_df = tabulate_results(results)
results_df

In [ ]:
long_results_df = results_df.iloc[:, :4].stack().rename("score").reset_index().rename(columns={"level_0": "dataset", "level_1": "metric"})

In [ ]:
g = sns.FacetGrid(long_results_df, col="metric")
g.map(sns.scatterplot, "dataset", "score")
[ax.tick_params(axis="x", rotation=45) for ax in g.axes[0, :]]

In [ ]:
# g.figure.savefig("../reports/figures/score_vis/score_vis_demo.png", bbox_inches="tight")

### Score individual file

In [ ]:
REF_FILE = "../data/processed/icdar_en_trial/hipe-ocrepair-bench_v0.9_icdar2017_v1.1_train_en_10k_sample.jsonl"
HYP_FILE = "../data/processed/icdar_en_trial/qwen35.jsonl"

In [ ]:
file_results = score_file_pair(REF_FILE, HYP_FILE)

In [ ]:
file_results["averaged_scores"]

In [ ]:
file_results["fold_scores"]

In [ ]:
tabulate_results(file_results)

### Score ICDAR/Impresso/DTA trial

In [ ]:
trial_name = "10k_sample_trial"
dataset = "dta"
lang = "de"

reference = "hipe-ocrepair-bench_v0.9_icdar2017_v1.1_train_fr_10k_sample.jsonl"

proc_folder = "../data/processed/"
figures_folder = "../reports/figures"

trial_folder = os.path.join(proc_folder, trial_name)
trial_figures = os.path.join(figures_folder, trial_name)

In [ ]:
folders = glob.glob(os.path.join(trial_folder, "*/*"))
folders

In [ ]:
results = {}
for folder in folders:
    dataset, subset = folder.split("\\")[1:3]
    results[f"{dataset}_{subset}"] = {}
    json_files = glob.glob(os.path.join(folder, "*.jsonl"))
    
    for j in json_files:
        if "10k" in os.path.basename(j):
            ref = j
            break
            
    json_files.remove(ref)

    for j in json_files:
        if "gemma3" in j:
            model = "gemma3"
        elif "gemma4" in j:
            model = "gemma4"
        elif "deepseek" in j:
            model = "deepseek"
        elif "mistral" in j:
            model = "mistral"
        elif "qwen" in j:
            model = "qwen"
        else:
            raise ValueError("Model not found")

        results[f"{dataset}_{subset}"][model] = score_file_pair(ref=ref, hyp=j)["averaged_scores"]

In [ ]:
dfs = []
for dataset, scores in results.items():
    df = pd.DataFrame(scores).T.rename_axis("model")
    df["dataset"] = dataset
    df.set_index("dataset", append=True, inplace=True)
    dfs.append(df.reorder_levels(["dataset", "model"]))

In [ ]:
trial_10k_df = tabulate_results(pd.concat(dfs))

In [ ]:
# icdar_results = {}
# for f in ICDAR_HYP:
#     model = f.split(".")[-2].split("_")[-1]
#     icdar_results[model] = score_file_pair(ICDAR_REF, f)["averaged_scores"]

# icdar_results_df = tabulate_results(icdar_results)
# icdar_results_df.to_csv(os.path.join(trial_folder, "scored_results.csv"))

In [ ]:
trial_10k_df.to_csv(os.path.join(trial_folder, "scored_results.csv"))

In [ ]:
trial_10k_long_df = trial_10k_df.loc[:, [
    "cmer_micro", "wmer_micro", "cmer_macro", "wmer_macro", "pref_score_cmer_macro", "pref_score_wmer_macro", "pcis_cmer_macro", "pcis_wmer_macro"
]].stack().rename("score").reset_index().rename(columns={"level_0": "dataset", "level_2": "metric"})

In [ ]:
def typeset_title(ax):
    title = ax.title.get_text()
    dataset, metric = title.split("|")
    dataset = dataset.split("=")[1].strip()
    metric = metric.split("=")[1].strip()
    ax.set_title(f"{dataset} | {metric}")
    return None

In [ ]:
cw_df = trial_10k_long_df.query("metric in ['cmer_micro', 'wmer_micro', 'cmer_macro', 'wmer_macro']")
cw_g = sns.FacetGrid(cw_df, col="metric", row="dataset", ylim=(0,0.2))
cw_g.map(sns.scatterplot, "model", "score")
plot_errorbars(plot_df=cw_df, results_df=trial_10k_df, axes=cw_g.axes.flatten())

[ax.tick_params(axis="x", rotation=45) for ax in cw_g.axes[-1, :]]
[typeset_title(ax) for ax in cw_g.axes.flatten()]

In [ ]:
pp_df = trial_10k_long_df.query("metric in ['pref_score_cmer_macro', 'pref_score_wmer_macro', 'pcis_cmer_macro', 'pcis_wmer_macro']")
pp_g = sns.FacetGrid(pp_df, col="metric", row="dataset", ylim=(0,1))
pp_g.map(sns.scatterplot, "model", "score")

plot_errorbars(plot_df=cw_df, results_df=trial_10k_df, axes=pp_g.axes.flatten())

[ax.tick_params(axis="x", rotation=45) for ax in pp_g.axes[-1, :]]
[typeset_title(ax) for ax in pp_g.axes.flatten()]

In [ ]:
cw_g.figure.savefig(os.path.join(trial_figures, "cmer_wmer.png"), bbox_inches="tight")
pp_g.figure.savefig(os.path.join(trial_figures, "pref_pcis.png"), bbox_inches="tight")

### Get individual statistics

In [ ]:
icdar_results_df[['cmer_micro', 'wmer_micro', 'cmer_macro', 'wmer_macro', 'pref_score_cmer_macro', 'pref_score_wmer_macro', 'pcis_cmer_macro', 'pcis_wmer_macro']]

In [ ]:
icdar_results_df[['cmer_micro', 'pref_score_cmer_macro']]

In [ ]:
icdar_results_df[['pref_score_cmer_macro', 'pref_score_cmer_macro_5ci', 'pref_score_cmer_macro_95ci']]

### Combine several sets of results

In [ ]:
df0 = pd.read_csv("..\\data\\processed\\icdar_fr_trial\\scored_results.csv", index_col=0)
df1 = pd.read_csv("..\\data\\processed\\icdar_en_trial\\scored_results.csv", index_col=0)

In [ ]:
combined_results_df = pd.concat([df0, df1])

In [ ]:
combined_results_df

In [ ]:
combined_long_df = combined_results_df.loc[:, [
    "cmer_micro", "wmer_micro", "cmer_macro", "wmer_macro", "pref_score_cmer_macro", "pref_score_wmer_macro", "pcis_cmer_macro", "pcis_wmer_macro"
]].stack().rename("score").reset_index().rename(columns={"level_0": "dataset", "level_1": "metric"})

In [ ]:
combined_cw_df = combined_long_df.query("metric in ['cmer_micro', 'wmer_micro', 'cmer_macro', 'wmer_macro']")
combined_cw_g = sns.FacetGrid(combined_cw_df, col="metric", ylim=(0,0.2))
combined_cw_g.map(sns.scatterplot, "dataset", "score")
[ax.tick_params(axis="x", rotation=45) for ax in combined_cw_g.axes[0, :]]
plot_errorbars(plot_df=combined_cw_df, results_df=combined_results_df, axes=combined_cw_g.axes[0, :])

In [ ]:
combined_pp_df = combined_long_df.query("metric in ['pref_score_cmer_macro', 'pref_score_wmer_macro', 'pcis_cmer_macro', 'pcis_wmer_macro']")
combined_pp_g = sns.FacetGrid(combined_pp_df, col="metric", ylim=(0,1))
combined_pp_g.map(sns.scatterplot, "dataset", "score")
[ax.tick_params(axis="x", rotation=45) for ax in combined_pp_g.axes[0, :]]
plot_errorbars(plot_df=combined_pp_df, results_df=combined_results_df, axes=combined_pp_g.axes[0, :])

In [ ]:
combined_cw_g.figure.savefig(os.path.join(trial_figures, "combined_cmer_wmer.png"), bbox_inches="tight")
combined_pp_g.figure.savefig(os.path.join(trial_figures, "combined_pref_pcis.png"), bbox_inches="tight")